In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["HADOOP_HOME"] = r"C:\hadoop" 
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.path.pathsep + os.environ["PATH"]

def create_spark_session():
    WAREHOUSE_DIR = os.path.abspath("output/iceberg_warehouse")
    return SparkSession.builder \
        .appName("RetailLakehouse") \
        .master("local[*]") \
        .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2") \
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
        .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
        .config("spark.sql.catalog.local.type", "hadoop") \
        .config("spark.sql.catalog.local.warehouse", WAREHOUSE_DIR) \
        .config("spark.sql.defaultCatalog", "local") \
        .getOrCreate()

spark = create_spark_session()

print("======================================================")
print("SUCCESS: Spark Session connected successfully!")
print(f"Spark Version: {spark.version}")
print(f"Warehouse Location: {os.path.abspath('output/iceberg_warehouse')}")
print("======================================================")


SUCCESS: Spark Session connected successfully!
Spark Version: 3.5.4
Warehouse Location: c:\Users\palisha.shakya\Downloads\Project1\Iceberg\output\iceberg_warehouse


In [2]:
from pyspark.sql.functions import col

# Load
df = spark.read.option("header", "true").csv("../Data/fact_sales.csv")

# Transform
df_transformed = df.select(
    col("order_id").cast("long"),
    col("order_date").cast("date"),
    col("customer_id").cast("long"),
    col("product_id").cast("long"),
    col("store_id").cast("long"),
    col("quantity").cast("long"),
    col("total_amount").cast("double")
)

# Write to Iceberg
df_transformed.write.mode("overwrite").saveAsTable("local.db.fact_sales")
print("Data loaded to Iceberg successfully.")

Data loaded to Iceberg successfully.


In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS local.db")
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.db.fact_sales (
        order_id bigint, 
        order_date date, 
        customer_id bigint, 
        product_id bigint, 
        store_id bigint, 
        quantity bigint, 
        total_amount double
    ) 
    USING iceberg 
    PARTITIONED BY (days(order_date))
""")

DataFrame[]

In [4]:
from pyspark.sql.functions import col

# ------------------------------------------------------------------
# 1. Read CSV
# ------------------------------------------------------------------
df = spark.read.option("header", "true").csv("../Data/fact_sales.csv")

# ------------------------------------------------------------------
# 2. Transform data types
# ------------------------------------------------------------------
df_transformed = df.select(
    col("order_id").cast("bigint").alias("order_id"),
    col("order_date").cast("date").alias("order_date"),
    col("customer_id").cast("bigint").alias("customer_id"),
    col("product_id").cast("bigint").alias("product_id"),
    col("store_id").cast("bigint").alias("store_id"),
    col("quantity").cast("bigint").alias("quantity"),
    col("total_amount").cast("double").alias("total_amount")
)

# ------------------------------------------------------------------
# 3. Remove existing unpartitioned table
# ------------------------------------------------------------------
spark.sql("DROP TABLE IF EXISTS local.db.fact_sales")

# ------------------------------------------------------------------
# 4. Create a NEW partitioned Iceberg table
# ------------------------------------------------------------------
spark.sql("""
CREATE TABLE local.db.fact_sales (
    order_id BIGINT,
    order_date DATE,
    customer_id BIGINT,
    product_id BIGINT,
    store_id BIGINT,
    quantity BIGINT,
    total_amount DOUBLE
)
USING iceberg
PARTITIONED BY (days(order_date))
""")

# ------------------------------------------------------------------
# 5. Load data into the table
# ------------------------------------------------------------------
df_transformed.writeTo("local.db.fact_sales").append()

print("Partitioned Iceberg table created successfully!")

Partitioned Iceberg table created successfully!


In [5]:
spark.sql("DESCRIBE EXTENDED local.db.fact_sales").show(200, truncate=False)

+----------------------------+----------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                             |comment|
+----------------------------+----------------------------------------------------------------------------------------------------------------------+-------+
|order_id                    |bigint                                                                                                                |NULL   |
|order_date                  |date                                                                                                                  |NULL   |
|customer_id                 |bigint                                                                                                                |NULL   |
|product_id                  |bigint                

In [6]:
spark.sql("""
EXPLAIN
SELECT *
FROM local.db.fact_sales
WHERE order_date = DATE '2026-06-29'
""").show(truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|plan                                                                                                                                                                                                                                                                                                                                                                  |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [7]:
spark.sql("""
SELECT file_path, partition
FROM local.db.fact_sales.files
""").show(truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+
|file_path                                                                                                                                                                            |partition   |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+
|c:/Users/palisha.shakya/Downloads/Project1/Iceberg/output/iceberg_warehouse/db/fact_sales/data/order_date_day=2026-09-16/00000-4-ada54cfe-af5f-44ec-980c-6850177c50ce-0-00052.parquet|{2026-09-16}|
|c:/Users/palisha.shakya/Downloads/Project1/Iceberg/output/iceberg_warehouse/db/fact_sales/data/order_date_day=2026-09-17/00000-4-ada54cfe-af5f-44ec-980c-6850177c50ce-0-00119.parquet|{2026-09-17}|
|c:/Users/palis

In [8]:
spark.sql("""
SELECT order_date, COUNT(*)
FROM local.db.fact_sales
GROUP BY order_date
ORDER BY order_date
""").show()

+----------+--------+
|order_date|count(1)|
+----------+--------+
|2026-01-02|       2|
|2026-01-04|       3|
|2026-01-05|       3|
|2026-01-06|       2|
|2026-01-07|       1|
|2026-01-08|       2|
|2026-01-09|       3|
|2026-01-10|       2|
|2026-01-11|       3|
|2026-01-12|       2|
|2026-01-13|       1|
|2026-01-14|       2|
|2026-01-15|       5|
|2026-01-16|       4|
|2026-01-17|       4|
|2026-01-18|       2|
|2026-01-19|       3|
|2026-01-20|       1|
|2026-01-21|       2|
|2026-01-22|       1|
+----------+--------+
only showing top 20 rows



In [9]:
spark.sql("""
SELECT COUNT(DISTINCT order_date)
FROM local.db.fact_sales
""").show()

+--------------------------+
|count(DISTINCT order_date)|
+--------------------------+
|                       335|
+--------------------------+



In [10]:
spark.sql("""
SELECT
partition,
record_count,
file_path
FROM local.db.fact_sales.files
ORDER BY partition
""").show(100, truncate=False)

+------------+------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|partition   |record_count|file_path                                                                                                                                                                            |
+------------+------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{2026-01-02}|2           |c:/Users/palisha.shakya/Downloads/Project1/Iceberg/output/iceberg_warehouse/db/fact_sales/data/order_date_day=2026-01-02/00000-4-ada54cfe-af5f-44ec-980c-6850177c50ce-0-00040.parquet|
|{2026-01-04}|3           |c:/Users/palisha.shakya/Downloads/Project1/Iceberg/output/iceberg_warehouse/db/fact_sales/data/order_date_day=2026-01-04/00000-4-ada5

In [11]:
spark.sql("""
SELECT COUNT(*)
FROM local.db.fact_sales.files
""").show()

+--------+
|count(1)|
+--------+
|     335|
+--------+



In [12]:
spark.sql("""
EXPLAIN
SELECT *
FROM local.db.fact_sales
WHERE order_date = DATE '2026-06-29'
""").show(truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|plan                                                                                                                                                                                                                                                                                                                                                                  |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [13]:
spark.sql("""
EXPLAIN FORMATTED
SELECT *
FROM local.db.fact_sales
WHERE order_date = DATE '2026-06-29'
""").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|plan                                                                                                                                                                                                                                                                 

In [15]:
spark.sql("""
DESCRIBE EXTENDED local.db.fact_sales
""").show(200, False)

+----------------------------+----------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                             |comment|
+----------------------------+----------------------------------------------------------------------------------------------------------------------+-------+
|order_id                    |bigint                                                                                                                |NULL   |
|order_date                  |date                                                                                                                  |NULL   |
|customer_id                 |bigint                                                                                                                |NULL   |
|product_id                  |bigint                

In [17]:
spark.sql("""
SELECT COUNT(*)
FROM local.db.fact_sales
""").show()

+--------+
|count(1)|
+--------+
|    1000|
+--------+



In [18]:
spark.sql("""
SELECT
MIN(order_date),
MAX(order_date)
FROM local.db.fact_sales
""").show()

+---------------+---------------+
|min(order_date)|max(order_date)|
+---------------+---------------+
|     2026-01-02|     2026-12-31|
+---------------+---------------+



In [19]:
spark.sql("""
SELECT
order_date,
COUNT(*) AS sales
FROM local.db.fact_sales
GROUP BY order_date
ORDER BY order_date
""").show(20)

+----------+-----+
|order_date|sales|
+----------+-----+
|2026-01-02|    2|
|2026-01-04|    3|
|2026-01-05|    3|
|2026-01-06|    2|
|2026-01-07|    1|
|2026-01-08|    2|
|2026-01-09|    3|
|2026-01-10|    2|
|2026-01-11|    3|
|2026-01-12|    2|
|2026-01-13|    1|
|2026-01-14|    2|
|2026-01-15|    5|
|2026-01-16|    4|
|2026-01-17|    4|
|2026-01-18|    2|
|2026-01-19|    3|
|2026-01-20|    1|
|2026-01-21|    2|
|2026-01-22|    1|
+----------+-----+
only showing top 20 rows



In [20]:
df = spark.sql("""
SELECT *
FROM local.db.fact_sales
""")

df.explain(True)

== Parsed Logical Plan ==
'Project [*]
+- 'UnresolvedRelation [local, db, fact_sales], [], false

== Analyzed Logical Plan ==
order_id: bigint, order_date: date, customer_id: bigint, product_id: bigint, store_id: bigint, quantity: bigint, total_amount: double, discount_applied: double
Project [order_id#658L, order_date#659, customer_id#660L, product_id#661L, store_id#662L, quantity#663L, total_amount#664, discount_applied#665]
+- SubqueryAlias local.db.fact_sales
   +- RelationV2[order_id#658L, order_date#659, customer_id#660L, product_id#661L, store_id#662L, quantity#663L, total_amount#664, discount_applied#665] local.db.fact_sales local.db.fact_sales

== Optimized Logical Plan ==
RelationV2[order_id#658L, order_date#659, customer_id#660L, product_id#661L, store_id#662L, quantity#663L, total_amount#664, discount_applied#665] local.db.fact_sales

== Physical Plan ==
*(1) ColumnarToRow
+- BatchScan local.db.fact_sales[order_id#658L, order_date#659, customer_id#660L, product_id#661L, sto

In [21]:
df = spark.sql("""
SELECT *
FROM local.db.fact_sales
WHERE order_date='2026-09-16'
""")

df.explain(True)

== Parsed Logical Plan ==
'Project [*]
+- 'Filter ('order_date = 2026-09-16)
   +- 'UnresolvedRelation [local, db, fact_sales], [], false

== Analyzed Logical Plan ==
order_id: bigint, order_date: date, customer_id: bigint, product_id: bigint, store_id: bigint, quantity: bigint, total_amount: double, discount_applied: double
Project [order_id#687L, order_date#688, customer_id#689L, product_id#690L, store_id#691L, quantity#692L, total_amount#693, discount_applied#694]
+- Filter (order_date#688 = cast(2026-09-16 as date))
   +- SubqueryAlias local.db.fact_sales
      +- RelationV2[order_id#687L, order_date#688, customer_id#689L, product_id#690L, store_id#691L, quantity#692L, total_amount#693, discount_applied#694] local.db.fact_sales local.db.fact_sales

== Optimized Logical Plan ==
Filter (order_date#688 = 2026-09-16)
+- RelationV2[order_id#687L, order_date#688, customer_id#689L, product_id#690L, store_id#691L, quantity#692L, total_amount#693, discount_applied#694] local.db.fact_sales



In [16]:
# View the first 5 rows 
spark.sql("SELECT * FROM local.db.fact_sales LIMIT 5").show()

+--------+----------+-----------+----------+--------+--------+------------+----------------+
|order_id|order_date|customer_id|product_id|store_id|quantity|total_amount|discount_applied|
+--------+----------+-----------+----------+--------+--------+------------+----------------+
|      13|2026-09-16|         89|        12|       1|       4|      595.68|            NULL|
|     239|2026-09-16|         95|        19|       3|       1|      155.64|            NULL|
|     263|2026-09-16|         71|        12|       2|       4|      233.64|            NULL|
|     860|2026-09-16|         94|        10|       2|       4|     1642.96|            NULL|
|     864|2026-09-16|         28|        18|       1|       4|      387.16|            NULL|
+--------+----------+-----------+----------+--------+--------+------------+----------------+



In [22]:
spark.sql("""
SELECT *
FROM local.db.fact_sales
WHERE order_date='2026-09-16'
""").show()

+--------+----------+-----------+----------+--------+--------+------------+----------------+
|order_id|order_date|customer_id|product_id|store_id|quantity|total_amount|discount_applied|
+--------+----------+-----------+----------+--------+--------+------------+----------------+
|      13|2026-09-16|         89|        12|       1|       4|      595.68|            NULL|
|     239|2026-09-16|         95|        19|       3|       1|      155.64|            NULL|
|     263|2026-09-16|         71|        12|       2|       4|      233.64|            NULL|
|     860|2026-09-16|         94|        10|       2|       4|     1642.96|            NULL|
|     864|2026-09-16|         28|        18|       1|       4|      387.16|            NULL|
+--------+----------+-----------+----------+--------+--------+------------+----------------+



In [24]:
spark.sql("""
INSERT INTO local.db.fact_sales
VALUES (
999999,
DATE '2026-10-01',
1,
1,
1,
2,
200.00,
15.00
)
""")

DataFrame[]

In [25]:
spark.sql("""
SELECT *
FROM local.db.fact_sales
WHERE order_id=999999
""").show()

+--------+----------+-----------+----------+--------+--------+------------+----------------+
|order_id|order_date|customer_id|product_id|store_id|quantity|total_amount|discount_applied|
+--------+----------+-----------+----------+--------+--------+------------+----------------+
|  999999|2026-10-01|          1|         1|       1|       2|       200.0|            15.0|
+--------+----------+-----------+----------+--------+--------+------------+----------------+



In [26]:
spark.sql("""
SELECT order_id,
discount_applied
FROM local.db.fact_sales
WHERE order_id IN (13,999999)
""").show()

+--------+----------------+
|order_id|discount_applied|
+--------+----------------+
|  999999|            15.0|
|      13|            NULL|
+--------+----------------+



In [14]:
# Execute the ALTER command to add the column
spark.sql("ALTER TABLE local.db.fact_sales ADD COLUMN discount_applied DOUBLE")

# Verify the schema change
spark.sql("DESCRIBE TABLE local.db.fact_sales").show()

+----------------+----------------+-------+
|        col_name|       data_type|comment|
+----------------+----------------+-------+
|        order_id|          bigint|   NULL|
|      order_date|            date|   NULL|
|     customer_id|          bigint|   NULL|
|      product_id|          bigint|   NULL|
|        store_id|          bigint|   NULL|
|        quantity|          bigint|   NULL|
|    total_amount|          double|   NULL|
|discount_applied|          double|   NULL|
|                |                |       |
|  # Partitioning|                |       |
|          Part 0|days(order_date)|       |
+----------------+----------------+-------+



Time Travel

In [27]:
spark.sql("""
SELECT *
FROM local.db.fact_sales.snapshots
""").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                                                                          |summary                                                                                                                                                                                                         

In [32]:
spark.sql("""SELECT COUNT(*)FROM local.db.fact_sales VERSION AS OF 6601570308674708664 """).show(truncate=False)

+--------+
|count(1)|
+--------+
|1000    |
+--------+



In [33]:
spark.sql("""
SELECT COUNT(*) AS total_orders
FROM local.db.fact_sales
""").show()

+------------+
|total_orders|
+------------+
|        1001|
+------------+



In [34]:
spark.sql("""
SELECT COUNT(*) AS total_orders
FROM local.db.fact_sales
VERSION AS OF 6601570308674708664
""").show()

+------------+
|total_orders|
+------------+
|        1000|
+------------+



In [35]:
current = spark.sql("""
SELECT COUNT(*) AS current_orders
FROM local.db.fact_sales
""")

old = spark.sql("""
SELECT COUNT(*) AS old_orders
FROM local.db.fact_sales
VERSION AS OF 6601570308674708664
""")

current.show()
old.show()

+--------------+
|current_orders|
+--------------+
|          1001|
+--------------+

+----------+
|old_orders|
+----------+
|      1000|
+----------+



In [36]:
spark.sql("""
SELECT *
FROM local.db.fact_sales
WHERE order_id = 999999
""").show()

+--------+----------+-----------+----------+--------+--------+------------+----------------+
|order_id|order_date|customer_id|product_id|store_id|quantity|total_amount|discount_applied|
+--------+----------+-----------+----------+--------+--------+------------+----------------+
|  999999|2026-10-01|          1|         1|       1|       2|       200.0|            15.0|
+--------+----------+-----------+----------+--------+--------+------------+----------------+



In [37]:
spark.sql("""
SELECT *
FROM local.db.fact_sales
VERSION AS OF 6601570308674708664
WHERE order_id = 999999
""").show()

+--------+----------+-----------+----------+--------+--------+------------+
|order_id|order_date|customer_id|product_id|store_id|quantity|total_amount|
+--------+----------+-----------+----------+--------+--------+------------+
+--------+----------+-----------+----------+--------+--------+------------+



This clearly demonstrates the row did not exist in the earlier snapshot.